In [1]:
import os
import conllu
#from conllu import parse
import spacy


In [2]:
# if using vectors
#import spacy
#from spacy.language import Language
#spacy.prefer_gpu()
#nlp = spacy.load("grc_proiel_trf")
#!python -m spacy init vectors grc ../assets/grc_floret_cbow_nn2_xn10_b200k_dim300.floret ../vectors/large --mode floret

Normalization is very important. We normalize the texts and their tags (lemmas, parts of speech, etc.).
We do that so that the dataset is unified and less variations, as well as clean up issues.<br>
The normalize_text() function provides configurable text normalization for Greek textual processing pipelines.<br> 

Key features include:

**Unicode normalization**: Supports NFC/NFD/NFKC/NFKD forms for canonical equivalence<br>
**Diacritic removal**: Optionally strips accents while preserving Greek apostrophes<br>
**Case normalization**: Optional lowercase conversion<br>
**Apostrophe standardization**: Unifies 5+ apostrophe variants to U+02BC (ʼ)<br>
**Structural cleaning**: Removes brackets/stray numbers/extra whitespace<br>
**Debug mode**: Shows before-after states for each processing step<br>
The function's modular design allows selective activation of normalization stages through boolean parameters, making it adaptable for different NLP preprocessing requirements in ancient Greek text analysis workflows.

In [9]:
import re
import unicodedata

# apostrophes and correct_apostrophe are defined as follows:
apostrophes = ["᾽", "᾿", "'", "’", "‘"]
correct_apostrophe = "ʼ"

def clean_and_remove_accents(text: str) -> str:
    """
    Cleans the given text by removing diacritics (accents), except for specific characters,
    and converting it to lowercase.
    """
    allowed_characters = [' ̓', "᾿", "᾽", "'", "’", "‘", 'ʼ', '̓']  # Including the Greek apostrophe
    if not isinstance(text, str):
        raise ValueError("Input must be a string.")
    try:
        non_accent_chars = [c for c in unicodedata.normalize('NFC', text) 
        if unicodedata.category(c) != 'Mn' or c in allowed_characters]
        return ''.join(non_accent_chars)
    
    except Exception as e:
        # A more generic exception handling if unexpected errors occur
        print(f"An error occurred: {e}")
        return text
    

def normalize_text(text: str, form: str = 'NFC', 
                   remove_accents: bool = False, 
                   lowercase: bool = False, 
                   standardize_apostrophe: bool = True, 
                   remove_brackets: bool = False, 
                   remove_trailing_numbers: bool = False, 
                   remove_extra_spaces: bool = False, 
                   debug: bool = False) -> str:
    """
    Applies multiple text normalization and cleaning steps on the input text.

    Parameters:
    - text (str): The text to be normalized.
    - form (str): Unicode normalization form ('NFC', 'NFD', 'NFKC', 'NFKD').
    - lowercase (bool): If True, the text is converted to lowercase.
    - standardize_apostrophe (bool): If True, replaces all defined apostrophe characters with a standard one.
    - remove_brackets_only (bool): If True, removes the brackets themselves.
    - remove_trailing_numbers (bool): If True, strips leading or trailing digits from the text.
    
    Returns:
    - str: The processed text.
    """
    normalized_text = text  # Initialize normalized_text with the original text

    # Function to print before and after states for each operation during debugging
    def debug_print(operation_name, before, after):
        if debug:
            print(f"{operation_name} - Before: {before}")
            print(f"{operation_name} - After: {after}")

    # Standardize apostrophe characters if required
    if standardize_apostrophe:
        before_text = normalized_text
        # Create translation table and apply replacements
        apostrophe_map = {ord(apos): ord(correct_apostrophe) for apos in apostrophes}
        normalized_text = normalized_text.translate(apostrophe_map)
        
        # Debug output if enabled
        debug_print("Standardizing apostrophes", before_text, normalized_text)
        
    if remove_accents:
        before_text = normalized_text
        try:
            normalized_text = clean_and_remove_accents(normalized_text)
        except Exception as e:
            print(f"An error occurred while removing accents: {e}")
            # Decide what to do here: return the original text, a special value, or stop the process
            return text        
        debug_print("Removing accents", before_text, normalized_text)
        
    # Convert to lowercase if required
    if lowercase:
        before_text = normalized_text
        normalized_text = normalized_text.lower()
        debug_print("Lowercase conversion", before_text, normalized_text)

    # Unicode normalization
    if form:
        before_text = normalized_text
        normalized_text = unicodedata.normalize(form, normalized_text)
        debug_print("Unicode normalization", before_text, normalized_text)
            
    # Remove brackets only if required
    if remove_brackets:
        before_text = normalized_text
        normalized_text = re.sub(r'[\(\)\[\]]', '', normalized_text)
        debug_print("Removing brackets", before_text, normalized_text)
        
    # Remove trailing numbers if required
    if remove_trailing_numbers:
        before_text = normalized_text
        normalized_text = re.sub(r'^\d+|\d+$', '', normalized_text)
        debug_print("Removing trailing numbers", before_text, normalized_text)

    # Remove multiple spaces and leading/trailing spaces
    if remove_extra_spaces:
        before_text = normalized_text
        normalized_text = ' '.join(normalized_text.split()).strip()
        debug_print("Removing extra spaces", before_text, normalized_text)

    return normalized_text

## Processing conllu files  
Step 1: Normalize the conllu files, which include various texts.
IT will save the normalized files into a "processed" folder.

In [15]:
import conllu
import os

def process_sentences(input_files, output_file, combine=False, show_apostrophe_changes=False):
    """
    Processes .conllu files: cleans text, separates sentences based on punctuation,
    and optionally combines multiple .conllu files.

    Args:
        input_files (list): List of paths to input .conllu files.
        output_file (str): Path to the output .conllu file.
        combine (bool, optional): Whether to combine input files. Defaults to False.
        show_apostrophe_changes (bool): Whether to show apostrophe changes for the whole file.
    """
    all_sentences = []

    # Track apostrophe changes across all sentences
    total_apostrophe_changes = {}

    # Loop through each input file, whether combining or not
    for input_file in input_files:
        with open(input_file, "r", encoding="utf-8") as f:
            text = f.read()
            
        sentences = conllu.parse(text)
        if combine:
            all_sentences.extend(sentences)  # Combine sentences from all files
        else:
            all_sentences = sentences  # Use sentences from the current file only
            break  # Exit loop after the first file if not combining

    rebuilt_sentences = []
    sent_id = 1
    current_sentence_tokens = []
    token_id = 1  # Initialize token id for running numbers throughout each sentence

    for sentence in all_sentences:
        for token in sentence:
            # Clean the text for 'form' and 'lemma'
            before_form = token['form']
            before_lemma = token['lemma']

            token['form'] = normalize_text(token['form'], remove_accents=False, lowercase=False, standardize_apostrophe=True, remove_extra_spaces=True, debug=False)
            token['lemma'] = normalize_text(token['lemma'], remove_accents=False, lowercase=False, standardize_apostrophe=True, remove_extra_spaces=True, debug=False)
            
            # Track apostrophe changes
            if show_apostrophe_changes:
                for apos in apostrophes:
                    if apos != correct_apostrophe:
                        # Count changes in form
                        form_count = before_form.count(apos)
                        if form_count > 0:
                            total_apostrophe_changes[apos] = total_apostrophe_changes.get(apos, 0) + form_count
                        
                        # Count changes in lemma
                        lemma_count = before_lemma.count(apos)
                        if lemma_count > 0:
                            total_apostrophe_changes[apos] = total_apostrophe_changes.get(apos, 0) + lemma_count
                            
            # Update current token's id to ensure running numbers
            token['id'] = token_id
            
            # Add the token to the current sentence tokens and increment token_id
            current_sentence_tokens.append(token)
            token_id += 1            
            
            # Check if the current token is punctuation that indicates end of a sentence
            if token["form"] in [".", "·"]:
                # Append the current sentence tokens as a new TokenList to rebuilt_sentences
                metadata = {"sent_id": str(sent_id), "text": "NA"}
                rebuilt_sentences.append(conllu.TokenList(tokens=current_sentence_tokens, metadata=metadata))
                current_sentence_tokens = []  # Reset for the next sentence
                sent_id += 1
                token_id = 1  # Reset token id for the new sentence
                
    # Finalize the last sentence if it doesn't end with specified punctuation
    if current_sentence_tokens:
        metadata = {"sent_id": str(sent_id), "text": "NA"}
        rebuilt_sentences.append(conllu.TokenList(tokens=current_sentence_tokens, metadata=metadata))
        
    # Write rebuilt sentences to the output file
    with open(output_file, "w", encoding="utf-8") as out_f:
        for sentence in rebuilt_sentences:
            out_f.write(sentence.serialize())
            out_f.write("\n\n")  # Ensure correct .conllu formatting with blank lines

    # Print summary
    summary = "Combined" if combine else "Original"
    print(f"{summary} number of sentences from input files: {len(all_sentences)}")
    print(f"Rebuilt and separated {len(rebuilt_sentences)} sentences.\n")

    # Show apostrophe changes summary for the whole file
    if show_apostrophe_changes and total_apostrophe_changes:
        total_replaced = sum(total_apostrophe_changes.values())
        print(f"\nApostrophe changes for {os.path.basename(input_files[0])}:")
        for apos, count in total_apostrophe_changes.items():
            print(f"  {apos} -> {correct_apostrophe}: {count}")
        print(f"  Total replaced: {total_replaced}")

# Example usage for processing a single file
# process_sentences(["input_file.conllu"], "output_file.conllu", combine=False)

# Example usage for combining multiple files into one output file
# process_sentences(["input_file1.conllu", "input_file2.conllu"], "combined_output.conllu", combine=True)

In [ ]:
directory = "../assets/Lemmatization_training_files"
for entry in os.listdir(directory):
    full_path = os.path.join(directory, entry)
    if os.path.isfile(full_path) and entry.endswith(".conllu"):
        process_sentences([full_path], "../assets/Lemmatization_training_files/Processed/" + entry[:-7] + "_NFKD.conllu", show_apostrophe_changes=True)

In [ ]:
directory = "../assets/Lemmatization_training_files"
for entry in os.listdir(directory):
    full_path = os.path.join(directory, entry)
    if os.path.isfile(full_path) and entry.endswith(".conllu"):
        process_sentences([full_path], "../assets/Lemmatization_training_files/Processed/" + entry[:-7] + "_NFC.conllu", show_apostrophe_changes=True)

Step 2:
Adjust tokens in parsed sentences for spaCy's lemmatizer format requirements and set defaults for various token conditions.
The output will be conllu files in folder 'Processed/lemma_train'

In [ ]:
import os
import random
import unicodedata
import conllu
from pathlib import Path
def adjust_tokens_for_spacy(sentences, debug=False):
    """
    Adjusts tokens in parsed sentences for spaCy's trainable lemmatizer requirements
    and handles specific token conditions, setting appropriate defaults.
    """
    for sentence in sentences:
        for token in sentence:
            # Adjustments for forms and lemmas
            if token["form"] in ['', "_", '—', '-']:
                token["form"] = token["lemma"] if token["lemma"] not in ['', "_", '—', '-'] else "_"
                print(f"Adjusted form for token {token['id']}, form: {token['form']}") if debug else None
                
            if token["lemma"] in ['', "_", '—', '-']:
                token["lemma"] = "_"
                print(f"Adjusted lemma for token {token['id']}, lemma: {token['lemma']}") if debug else None
            
            # ID and UPOS adjustments
            if token["id"] == '':
                token["id"] = "UNK"  # Example arbitrary value for unknown IDs
                print(f"Adjusted ID for token {token['form']}, ID: {token['id']}") if debug else None
                
            if token["upos"] in ['', "_", '—', '-']:
                token["upos"] = "_"  # Use '' for as per spaCy standard or 'X' for unknown UPOS as per CoNLL-U standard
                print(f"Adjusted UPOS for token {token['form']}, UPOS: {token['upos']}") if debug else None
                
            if token["upos"] in ['END', 'MID']:
                token["upos"] = "NOUN"  # Correcting specific UPOS conditions
                print(f"Adjusted UPOS for token {token['form']}, UPOS: {token['upos']}") if debug else None

    return sentences

def process_and_normalize_files(input_directory, output_directory, normalization_form='NFC', debug=False):
    """
    Process .conllu files in the given directory, normalize text according
    to the specified normalization form, and split data into training and
    development sets, with conditions adjusted for spaCy's requirements.
    """
    Path(output_directory).mkdir(parents=True, exist_ok=True)
    # Check if input directory exists
    if not os.path.exists(input_directory):
        print(f"Error: The input directory '{input_directory}' does not exist.")
        return

    # Check if output directory exists, create if not
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        print(f"Created output directory: {output_directory}") if debug else None
    
    # Process and normalize each .conllu file in the input directory
    for file_name in os.listdir(input_directory):
        if file_name.endswith(".conllu"):
            print("\n", file_name) if debug else None
            
            # Read file
            file_path = os.path.join(input_directory, file_name)
            sentences = conllu.parse(open(file_path, "r", encoding="utf-8").read())
            
            # Adjust tokens before normalization and spaCy conversion
            sentences = adjust_tokens_for_spacy(sentences)
            
            # Parse sentences
            for sentence in sentences:
                for token in sentence:
                    # Apply normalization to token form and lemma
                    token["form"] = normalize_text(token["form"], form=normalization_form)
                    token["lemma"] = normalize_text(token["lemma"], form=normalization_form)
            
            # Set the seed for reproducibility
            random.seed(42)
            # Shuffle the sentences randomly
            random.shuffle(sentences)

            # Split the sentences into training and development data
            split_index = int(len(sentences) * 0.9)
            train_data, dev_data = sentences[:split_index], sentences[split_index:]
            
            # Write training and development data
            train_output_file = os.path.join(output_directory, f"{file_name[:-7]}_{normalization_form}_train.conllu")
            dev_output_file = os.path.join(output_directory, f"{file_name[:-7]}_{normalization_form}_dev.conllu")
            with open(train_output_file, "w", encoding="utf-8") as train_file:
                for sentence in train_data:
                    train_file.write(sentence.serialize())
            with open(dev_output_file, "w", encoding="utf-8") as dev_file:
                for sentence in dev_data:
                    dev_file.write(sentence.serialize())
            
            print(f"Processed and normalized {file_name}. Train and dev data saved.") if debug else None

# Example usage:
# Make sure to specify your actual paths for the input directory and output directory
# process_and_normalize_files("../assets/Lemmatization_training_files/test", "../assets/Lemmatization_training_files/lemma_train", "NFC")

In [ ]:
process_and_normalize_files("../assets/Lemmatization_training_files/Processed", "../assets/Lemmatization_training_files/Processed/lemma_train", "NFC", debug=True)

## Preparing spaCy Files

Now we convert the files to spaCy format files.  
We first validate head indices to make sure all the tokens are within valid range.  
We then read and parse each conllu file and convert it to spaCy file.  
The input is a directory (like the 'lemma_train' from before).  
The output will be in the directory you define below on run.

In [22]:
import os
import conllu
import subprocess

def validate_head_indices(sentences, debug=False):
    """
    Validates that all head indices in the tokens of the sentences are within the valid range.

    Args:
        sentences (List[TokenList]): List of sentences parsed from a .conllu file.

    Returns:
        bool: True if all head indices are valid, False otherwise.
    """
    for sentence in sentences:
        token_ids = {token["id"] for token in sentence}  # Set of valid token IDs for reference
        for token in sentence:
            # Assuming head is directly accessible in token and is an int
            head = token.get("head")  # Use .get() to safely handle missing 'head' entries
            
            # Check if head exists or is set to None
            if head is None:
                print(f"Missing head for token '{token['form']}' in sentence: {sentence.metadata.get('text', 'NA')}") if debug else None
                return False

            # Check if head index is within the valid range or is a root (0)
            if head not in token_ids and head != 0:
                print(f"Invalid head index {head} for token '{token['form']}' in sentence: {sentence.metadata.get('text', 'NA')}") if debug else None
                return False
    
    return True

def read_and_parse_conllu(file_path, debug=False):
    """
    Reads and parses a .conllu file from the given path.
    """
    
    Path(output_directory).mkdir(parents=True, exist_ok=True)
    # Check if input directory exists
    if not os.path.exists(input_directory):
        print(f"Error: The input directory '{input_directory}' does not exist.") if debug else None
        return

    # Check if output directory exists, create if not
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        print(f"Created output directory: {output_directory}") if debug else None

    with open(file_path, "r", encoding="utf-8") as data:
        annotations = data.read()
    return conllu.parse(annotations)

def convert_to_spacy(file_path, output_directory, sentences):
    """
    Converts the .conllu file to spaCy format if head indices are valid.
    """
    extra_args = "--n-sents 10" if len(sentences) >= 10 else ""
    convert_command = f"python -m spacy convert {file_path} {output_directory} -c conllu -m --merge-subtokens {extra_args}"
    
    try:
        subprocess.run(convert_command.split(), check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error converting file '{os.path.basename(file_path)}':", e) # Print error message

def process_conllu_files(input_directory, output_directory, debug=False):
    """
    Processes all .conllu files in the input directory, validating and converting them.
    """
    for file_name in os.listdir(input_directory):
        if file_name.endswith(".conllu"):
            file_path = os.path.join(input_directory, file_name)
            
            print(f"\nProcessing file: {file_name}") # Print current file being processed
            sentences = read_and_parse_conllu(file_path)
            
            #if validate_head_indices(sentences):
            #    print(f"{file_name}: {len(sentences)} sentences - Head indices valid.")
            #    convert_to_spacy(file_path, output_directory, sentences)
            #else:
            #    print(f"{file_name}: Head indices validation failed. Conversion skipped.")
            #convert_to_spacy(file_path, output_directory, sentences)
            convert_to_spacy(file_path, output_directory, sentences)
# Example usage
#input_directory = "../assets/Lemmatization_training_files/test/"
#output_directory = "../assets/Lemmatization_training_files/test/"
#process_conllu_files(input_directory, output_directory)

In [ ]:
input_directory = "../assets/Lemmatization_training_files/Processed/lemma_train/"
output_directory = "../assets/Lemmatization_training_files/Processed/lemma_train/spaCy/"
process_conllu_files(input_directory, output_directory, debug=True)

# Process Proiel and Perseus files
If you are using Proiel and/or Perseus files, the following code will adjust and convert them to spaCy format.

In [ ]:

# Process Proiel and Perseus files

def process_corpus_conllu_files(input_directory, output_directory, normalization_form):
    """
    Processes .conllu files in the given directory, normalizes token forms and lemmas 
    using the specified normalization form, and writes them to a new directory.
    """
    # Ensure output directory exists
    os.makedirs(output_directory, exist_ok=True)

    for file_name in os.listdir(input_directory):
        if file_name.endswith(".conllu"):
            input_file_path = os.path.join(input_directory, file_name)
            with open(input_file_path, "r", encoding="utf-8") as file:
                annotations = file.read()

            sentences = conllu.parse(annotations)
            output_file_path = os.path.join(output_directory, f"{file_name[:-7]}_{normalization_form}.conllu")
            
            with open(output_file_path, "w", encoding="utf-8") as file:
                for sentence in sentences:
                    for token in sentence:
                        token["form"] = normalize_text((token["form"]), normalization_form, remove_accents=False, lowercase=False, standardize_apostrophe=True, remove_extra_spaces=True, debug=False)
                        token["lemma"] = normalize_text(token["lemma"], normalization_form, remove_accents=False, lowercase=False, standardize_apostrophe=True, remove_extra_spaces=True, debug=False)
                    file.write(sentence.serialize())
            
            print(f"Processed file: {file_name}")
            token['form'] = normalize_text(token['form'], remove_accents=False, lowercase=False, standardize_apostrophe=True, remove_extra_spaces=True, debug=False)

# Example usage:
#input_directory = "../assets/UD_Ancient_Greek-Perseus"
#output_directory_nfc = "../assets/UD_Ancient_Greek-Perseus/UD_Ancient_Greek-Perseus_NFC"
#process_conllu_files(input_directory, output_directory_nfc, 'NFC')


Step 1: Process the files

In [ ]:
#Perseus conversion
input_directory = "../assets/UD_Ancient_Greek-Perseus"
output_directory_nfc = "../assets/UD_Ancient_Greek-Perseus/UD_Ancient_Greek-Perseus_NFC"
process_corpus_conllu_files(input_directory, output_directory_nfc, normalization_form='NFC')



In [ ]:
#Proiel conversion
input_directory = "../assets/UD_Ancient_Greek-PROIEL"
output_directory_nfc = "../assets/UD_Ancient_Greek-PROIEL/UD_Ancient_Greek-PROIEL_NFC"
process_corpus_conllu_files(input_directory, output_directory_nfc, normalization_form='NFC')

Step 2: convert to spaCy

In [ ]:
# convert conllu to spacy UD_Ancient_Greek and UD_Ancient_Greek-PROIEL
!python -m spacy convert ../assets/UD_Ancient_Greek-Perseus/UD_Ancient_Greek-Perseus_NFC/ ../assets/UD_Ancient_Greek-Perseus/UD_Ancient_Greek-Perseus_NFC -c conllu -m --n-sents 10 --merge-subtokens

!python -m spacy convert ../assets/UD_Ancient_Greek-PROIEL/UD_Ancient_Greek-PROIEL_NFC/ ../assets/UD_Ancient_Greek-PROIEL/UD_Ancient_Greek-PROIEL_NFC/ -c conllu -m --n-sents 10 --merge-subtokens



# Dataset tests

In [ ]:
import spacy
from spacy.tokens import DocBin

# install spacy grc model if not already installed
nlp = spacy.load("grc_proiel_trf") # Use your preferred model here

lemma_train= DocBin().from_disk('../corpus/train/lemma_train/train_lemma_NFKC.spacy')
lemma_train_docs = list(lemma_train.get_docs(nlp.vocab))

PROIEL_NFKD= DocBin().from_disk('../assets/UD_Ancient_Greek-PROIEL/UD_Ancient_Greek-PROIEL_NFKD/grc_proiel-ud-train_NFKD.spacy')
PROIEL_NFKD_docs = list(PROIEL_NFKD.get_docs(nlp.vocab))

PROIEL_NFKC= DocBin().from_disk('../assets/UD_Ancient_Greek-PROIEL/UD_Ancient_Greek-PROIEL_NFKC/grc_proiel-ud-train_NFKC.spacy')
PROIEL_NFKC_docs = list(PROIEL_NFKC.get_docs(nlp.vocab))

Perseus_NFKD= DocBin().from_disk('../assets/UD_Ancient_Greek-Perseus/UD_Ancient_Greek-Perseus_NFKD/grc_perseus-ud-train_NFKD.spacy')
Perseus_NFKD_docs = list(Perseus_NFKD.get_docs(nlp.vocab))

Perseus_NFKC= DocBin().from_disk('../assets/UD_Ancient_Greek-Perseus/UD_Ancient_Greek-Perseus_NFKC/grc_perseus-ud-train_NFKC.spacy')
Perseus_NFKC_docs = list(Perseus_NFKC.get_docs(nlp.vocab))

In [ ]:
# iterate through sentences in lemma_train. If a sentence is in any of the other files, print the sentence and the file it is in
for doc in lemma_train_docs:
    if doc in PROIEL_NFKD_docs:
        print("PROIEL_NFKD")
        print(doc)
    if doc in PROIEL_NFKC_docs:
        print("PROIEL_NFKC")
        print(doc)
    if doc in Perseus_NFKD_docs:
        print("Perseus_NFKD")
        print(doc)
    if doc in Perseus_NFKC_docs:
        print("Perseus_NFKC")
        print(doc)
        